In [1]:
import os
import pandas as pd
import numpy as np

# 1. Генерация реалистичного датасета (чтобы было что анализировать)
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

np.random.seed(42)
n_samples = 100
data = {
    "id": range(1, n_samples + 1),
    "age": np.random.randint(18, 65, size=n_samples),
    "income": np.random.normal(50000, 15000, size=n_samples).astype(int),
    "category": np.random.choice(["A", "B", "C", None], size=n_samples, p=[0.4, 0.3, 0.2, 0.1]),
    "target": np.random.choice([0, 1], size=n_samples)
}
# Добавим немного дубликатов и выбросов для 3 лабы
df_generated = pd.DataFrame(data)
df_generated.loc[101] = df_generated.loc[10] # дубликат
df_generated.loc[102] = [103, 150, -5000, "X", 1] # выбросы и ошибки

df_generated.to_csv("../data/raw/dataset.csv", index=False)
df_generated.to_excel("../data/raw/dataset.xlsx", index=False)
df_generated.to_json("../data/raw/dataset.json", orient="records", force_ascii=False)
print("Файлы сгенерированы.\n---")

# 2. Загрузка CSV, Excel и JSON (Упр 2, 3, 4)
file_path = "../data/raw/dataset.csv"
df = pd.read_csv(file_path)
print(f"CSV загружен. Размер: {df.shape}")

excel_path = "../data/raw/dataset.xlsx"
df_excel = pd.read_excel(excel_path)
print(f"Excel загружен. Листы: {pd.ExcelFile(excel_path).sheet_names}")

json_path = "../data/raw/dataset.json"
df_json = pd.read_json(json_path)
print(f"JSON загружен. Типы:\n{df_json.dtypes}\n---")

# 3. Первичное изучение (Упр 6, 7)
print("Первые 3 строки:\n", df.head(3))
print("\nИнформация о наборе:")
df.info()

# 4. Анализ пропусков и дубликатов (Упр 8, 9)
missing_report = pd.DataFrame({"Пропущено": df.isna().sum(), "Доля, %": df.isna().mean() * 100})
print("\nОтчет по пропускам:\n", missing_report[missing_report["Пропущено"] > 0])
print(f"\nКоличество полных дубликатов: {df.duplicated().sum()}")

# 5. Описательная статистика и типы (Упр 10, 11)
print("\nСтатистика числовых данных:\n", df.describe().T)
print("\nКатегориальные данные:\n", df.select_dtypes(include=["object"]).value_counts(dropna=False))

# 6. Формирование отчета (Упр 14)
summary = {
    "Строк": df.shape[0],
    "Столбцов": df.shape[1],
    "Пропусков общ.": int(df.isna().sum().sum()),
    "Дубликатов": int(df.duplicated().sum())
}
pd.DataFrame(summary.items(), columns=["Показатель", "Значение"]).to_csv("../reports/data_quality_report.csv", index=False)
print("\nАвтоматический отчет сохранен в reports/data_quality_report.csv")

Файлы сгенерированы.
---
CSV загружен. Размер: (102, 5)
Excel загружен. Листы: ['Sheet1']
JSON загружен. Типы:
id           int64
age          int64
income       int64
category    object
target       int64
dtype: object
---
Первые 3 строки:
    id  age  income category  target
0   1   56   59544        A       0
1   2   46   36399      NaN       0
2   3   32   57140        B       1

Информация о наборе:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        102 non-null    int64 
 1   age       102 non-null    int64 
 2   income    102 non-null    int64 
 3   category  89 non-null     object
 4   target    102 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 4.1+ KB

Отчет по пропускам:
           Пропущено    Доля, %
category         13  12.745098

Количество полных дубликатов: 1

Статистика числовых данных:
         count          me

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

# 1. Загрузка и проверка (Упр 1, 2)
df = pd.read_csv("../data/raw/dataset.csv")
df_original = df.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)

# 2. Очистка дубликатов и пропусков (Упр 3-7)
df = df.drop_duplicates().reset_index(drop=True)
# Заполняем пропуски в категориях словом "Неизвестно"
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    df[col] = df[col].fillna("Неизвестно")

# 3. Проверка логических ограничений (Упр 10)
# Удаляем неадекватный возраст и отрицательный доход
invalid_mask = (df["age"] < 0) | (df["age"] > 120) | (df["income"] < 0)
df = df[~invalid_mask].reset_index(drop=True)

# 4. Обработка выбросов методом IQR (Упр 11, 12)
def clip_outliers_iqr(data, col):
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1
    data[col] = data[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)
    return data

for col in ["age", "income"]:
    df = clip_outliers_iqr(df, col)

# 5. Создание конвейера предобработки (Pipeline) (Упр 18)
numeric_features = ["age", "income"]
categorical_features = ["category"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Обучаем препроцессор на фичах (без target и id)
X = df.drop(columns=["id", "target"])
processed_array = preprocessor.fit_transform(X)
print("Размер массива после Pipeline:", processed_array.shape)

# 6. Сохранение результатов (Упр 19, 20)
Path("../models").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, "../models/preprocessor.joblib")
df.to_csv("../data/processed/dataset_clean.csv", index=False, encoding="utf-8-sig")

# 7. Сравнение (Упр 21)
comparison = pd.DataFrame({
    "Показатель": ["Строк", "Столбцов", "Пропусков", "Дубликатов"],
    "До": [df_original.shape[0], df_original.shape[1], int(df_original.isna().sum().sum()), int(df_original.duplicated().sum())],
    "После": [df.shape[0], df.shape[1], int(df.isna().sum().sum()), int(df.duplicated().sum())]
})
print("\nСравнение до и после очистки:\n", comparison)

Размер массива после Pipeline: (100, 6)

Сравнение до и после очистки:
    Показатель   До  После
0       Строк  102    100
1    Столбцов    5      5
2   Пропусков   13      0
3  Дубликатов    1      0


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Подготовка
reports_dir = Path("../reports/eda")
reports_dir.mkdir(parents=True, exist_ok=True)

# 1. Загрузка очищенных данных (Упр 1, 2)
df = pd.read_csv("../data/processed/dataset_clean.csv")
numeric_columns = ["age", "income"]
categorical_columns = ["category"]
target_column = "target"

# 2. Расчет статистики (Упр 3, 4)
desc_stats = df[numeric_columns].describe().T
desc_stats["skewness"] = df[numeric_columns].skew()
desc_stats.to_csv(reports_dir / "descriptive_statistics.csv")
print("Описательная статистика сохранена.")

# 3. Построение гистограмм (Упр 5)
for col in numeric_columns:
    plt.figure(figsize=(6, 4))
    plt.hist(df[col], bins=15, edgecolor='black')
    plt.title(f"Распределение: {col}")
    plt.savefig(reports_dir / f"hist_{col}.png")
    plt.close()

# 4. Диаграммы размаха и Bar plot (Упр 6, 7)
for col in numeric_columns:
    plt.figure(figsize=(6, 2))
    plt.boxplot(df[col], vert=False)
    plt.title(f"Boxplot: {col}")
    plt.savefig(reports_dir / f"boxplot_{col}.png")
    plt.close()

for col in categorical_columns:
    plt.figure(figsize=(6, 4))
    df[col].value_counts().plot(kind='bar')
    plt.title(f"Категории: {col}")
    plt.savefig(reports_dir / f"bar_{col}.png")
    plt.close()

# 5. Анализ целевой переменной (Упр 8)
plt.figure(figsize=(6, 4))
df[target_column].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title("Распределение Target")
plt.savefig(reports_dir / "target_distribution.png")
plt.close()

# 6. Матрица корреляций (Упр 9, 10)
corr_matrix = df[numeric_columns + [target_column]].corr()
corr_matrix.to_csv(reports_dir / "correlation_matrix.csv")

plt.figure(figsize=(6, 5))
plt.imshow(corr_matrix, cmap='coolwarm', aspect='auto')
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title("Корреляционная матрица")
plt.savefig(reports_dir / "correlation_heatmap.png")
plt.close()

# 7. Диаграммы рассеяния (Упр 12)
plt.figure(figsize=(6, 4))
plt.scatter(df["age"], df["income"], alpha=0.5, c=df["target"], cmap='bwr')
plt.title("Зависимость Income от Age (цветом - target)")
plt.xlabel("Age")
plt.ylabel("Income")
plt.savefig(reports_dir / "scatter_age_income.png")
plt.close()

print(f"Все графики (7 шт.) успешно сгенерированы и сохранены в {reports_dir}")

Описательная статистика сохранена.
Все графики (7 шт.) успешно сгенерированы и сохранены в ..\reports\eda
